# Compact LoRA with Soft-Balanced Sampling for Nemotron Reasoning


## Notebook Overview
This notebook demonstrates a simple and reproducible pipeline for the
Nemotron reasoning challenge using:
- 4-category task grouping
- Soft balance sampling
- Lightweight SFT (~600 samples)
- Compact LoRA adapter for submission  
The goal is not heavy training but a reproducible and competition-friendly pipeline.

**What this notebook does / does not do**
- does: simple category-aware sampling + lightweight LoRA
- does not: full hyperparameter search, synthetic data, multi-run ensembling

## 4-category task grouping
The dataset contains multiple types of reasoning tasks rather than a single uniform task.
To better balance the training data, I group prompts into four categories based on their dominant transformation pattern:
- Bit manipulation (binary transformations)
- String transformation (character or symbol mappings)
- Number / representation conversion (Roman numerals, unit conversions, etc.)
- Mathematical reasoning (arithmetic and symbolic operations)

This simple grouping captures the main task types in the dataset and allows us to apply soft-balanced sampling before lightweight LoRA fine-tuning.

In [1]:
!pip install -q --no-index --find-links /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages datasets trl --ignore-installed

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
torchvision 0.26.0.dev20260315+cu128 requires torch==2.12.0.dev20260315, but you have torch 2.10.0 which is incompatible.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
ydata-profiling 4.18.1 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
dopamine-rl 4.1.2 r

In [2]:
import os
import sys
import stat
import shutil
import re
import time
import random
import zipfile
import torch
import torch.nn.functional as F
import pandas as pd
import datasets
import trl
import kagglehub
import triton.backends.nvidia.compiler as nv_compiler
import triton.backends.nvidia as nv_backend

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM,TrainerCallback
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

print("datasets version:", datasets.__version__, "trl version:", trl.__version__)

/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


datasets version: 4.3.0 trl version: 0.24.0


In [3]:
# Copy PTXAS binaries to a writable temp directory
src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"

if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"] = dst

print(os.environ["TRITON_PTXAS_PATH"])

/tmp/ptxas-blackwell


In [4]:
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
nv_compiler.get_ptxas_version = lambda arch: "12.0"

def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

In [5]:
#Parameters
MAX_LORA_RANK = 32
MAX_SEQ_LEN = 1024 
NUM_EPOCHS = 1 
GRAD_ACCUM = 4 
LR = 2e-4 

SAMPLE_SIZE = 777
BALANCE_ALPHA=0.5  # 0 = original distribution, 1 = fully balanced
SEED = 777

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

OUTPUT_DIR = "/kaggle/working/lora_adapter"
ZIP_PATH = "/kaggle/working/submission.zip"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [6]:
train = pd.read_csv('/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv')
print("Train dataset loaded")
print("=== Dataset Summary ===")
print(f"Train shape: {train.shape}")
print(f"Number of prompts: {train.shape[0]}")
print(f"Number of columns: {train.shape[1]}")
train.head()

Train dataset loaded
=== Dataset Summary ===
Train shape: (9500, 3)
Number of prompts: 9500
Number of columns: 3


,id,prompt,answer
0,00066667,"In Alice's Wonderland, a secret bit manipulati...",10010111
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati...",01000011
2,00189f6a,"In Alice's Wonderland, secret encryption rules...",cat imagines book
3,001b24c4,"In Alice's Wonderland, numbers are secretly co...",XXXVIII
4,001c63cb,"In Alice's Wonderland, secret encryption rules...",wizard creates secret


In [7]:
def classify_prompt(text):
    """
    Classify a Nemotron reasoning prompt into one of four categories:
    bit, string, number, math.

    The classification is based on heuristic scoring using pattern matching.
    Each category receives points depending on detected patterns,
    and the category with the highest score is selected.

    Categories:
        bit    : binary / bit manipulation patterns
        string : character or symbol transformations
        number : numeric representation conversions (Roman numerals, units, etc.)
        math   : arithmetic or symbolic expressions

    This rule-based approach is simple, reproducible, and suitable for
    data-centric sampling strategies in lightweight SFT pipelines.
    """

    text_lower = text.lower()
    scores = {
        "bit": 0,
        "string": 0,
        "number": 0,
        "math": 0
    }

    # Bit manipulation patterns
    if re.search(r'[01]{6,}', text):
        scores["bit"] += 3
    if "bit" in text_lower or "binary" in text_lower:
        scores["bit"] += 2

    # String transformation patterns
    if "->" in text:
        scores["string"] += 3
    if re.search(r'[a-z]{3,}\s*->\s*[a-z]{3,}', text_lower):
        scores["string"] += 2
    if re.search(r'[!@#$%^&*()\[\]{}]', text):
        scores["string"] += 1

    # Number / representation conversion
    if re.search(r'\b[ivxlcdm]+\b', text_lower):
        scores["number"] += 3
    if re.search(r'\d+\.\d+', text):
        scores["number"] += 2
    if re.search(r'\d+\s*(cm|mm|kg|m)', text_lower):
        scores["number"] += 2

    # Math expressions
    if re.search(r'[\+\-\*/=]', text):
        scores["math"] += 2
    if re.search(r'\d+\s*[\+\-\*/]\s*\d+', text):
        scores["math"] += 2
    if any(op in text for op in ["@", "|", "!", "%"]):
        scores["math"] += 1

    # Select highest score
    return max(scores, key=scores.get)

train['category'] = train['prompt'].apply(classify_prompt)

In [8]:
def soft_balance_sample(df, total=600, alpha=0.5):
    """
    Sample a small training subset by interpolating between the
    original category distribution and a fully balanced distribution.

    alpha = 0 keeps the original distribution.
    alpha = 1 uses a fully balanced distribution.
    """
    
    counts = df['category'].value_counts()
    orig_ratio = counts / counts.sum()

    balanced = {c: 1/len(counts) for c in counts.index}
    target_ratio = (1-alpha)*orig_ratio + alpha*pd.Series(balanced)

    samples = []
    for c in counts.index:
        n = int(total * target_ratio[c])
        subset = df[df.category == c]
        samples.append(subset.sample(min(n, len(subset)), random_state=SEED))
    return pd.concat(samples)

sampled = soft_balance_sample(train, total=SAMPLE_SIZE, alpha=BALANCE_ALPHA)

def show_category_distribution(df, title):
    counts = df["category"].value_counts().sort_index()
    ratios = (counts / counts.sum()).round(3)

    dist_df = counts.to_frame(name="count")
    dist_df["ratio"] = ratios

    print(title)
    display(dist_df)

print("Train dataset sampled")
print("=== Sampling Summary ===")
print("\nOriginal dataset:")
print(f"Total original rows: {len(train)}")
show_category_distribution(train, "Original category distribution:")
print("\nSampled dataset:")
print(f"Total sampled rows: {len(sampled)}")
show_category_distribution(sampled, "Sampled category distribution:")

Train dataset sampled
=== Sampling Summary ===

Original dataset:
Total original rows: 9500
Original category distribution:


,count,ratio
category,,
bit,1602,0.169
math,1503,0.158
number,3191,0.336
string,3204,0.337



Sampled dataset:
Total sampled rows: 775
Sampled category distribution:


,count,ratio
category,,
bit,162,0.209
math,158,0.204
number,227,0.293
string,228,0.294


In [9]:
def format_row(row):
    return {
        "text": row["prompt"] + "\nAnswer:\n\\boxed{" + row["answer"] + "}"
    }

dataset = Dataset.from_pandas(sampled.apply(format_row, axis=1, result_type='expand'))

In [10]:
BASE_MODEL = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")

print("=== Model Loading ===")
start_time = time.time()

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True, 
    dtype=torch.bfloat16,
    device_map="auto"
)

elapsed = time.time() - start_time
print(f"Model and tokenizer loaded in {elapsed:.1f} seconds.")

=== Model Loading ===
Loading tokenizer...
Loading base model...


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Model and tokenizer loaded in 635.6 seconds.


In [11]:
for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}: is_fast_path_available = False")

Patched transformers_modules._1.modeling_nemotron_h: is_fast_path_available = False


In [12]:
print("=== LoRA Setup ===")
lora_config = LoraConfig(
    r=MAX_LORA_RANK,
    lora_alpha=16,
    target_modules="all-linear", #["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

=== LoRA Setup ===


/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 883,873,792 || all params: 32,461,811,136 || trainable%: 2.7228


In [13]:
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
import triton.backends.nvidia.compiler as nv_compiler
nv_compiler.get_ptxas_version = lambda arch: "12.0"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,           
    gradient_accumulation_steps=GRAD_ACCUM,  
    num_train_epochs=NUM_EPOCHS,             
    learning_rate=LR,                        
    logging_steps=1,                         
    bf16=True,                               
    max_grad_norm=1.0,                       
    optim="adamw_torch",                     
    lr_scheduler_type="cosine",              
    warmup_steps=0.1,                        
    save_strategy="no",                      
    report_to="none",
    dataset_text_field="text",               
    max_length=MAX_SEQ_LEN,              
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True}
)

class ProgressCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            print(
                f"[step {state.global_step}/{state.max_steps}] "
                f"loss={logs.get('loss')} "
                f"lr={logs.get('learning_rate')} "
                f"epoch={logs.get('epoch')}"
            )

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args,
    callbacks=[ProgressCallback()]
)


print("=== Training ===")
trainer.train()

Adding EOS to train dataset:   0%|          | 0/775 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/775 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/775 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 11, 'pad_token_id': 11}.


=== Training ===


Step,Training Loss
1,9.084606
2,15.247585
3,8.242448
4,15.111144
5,6.600117
6,8.555483
7,7.616472
8,7.056986
9,15.987354
10,9.526388


[step 1/194] loss=9.084606170654297 lr=0.0 epoch=0.005161290322580645
[step 2/194] loss=15.24758529663086 lr=1e-05 epoch=0.01032258064516129
[step 3/194] loss=8.242447853088379 lr=2e-05 epoch=0.015483870967741935
[step 4/194] loss=15.111144065856934 lr=3e-05 epoch=0.02064516129032258
[step 5/194] loss=6.600116729736328 lr=4e-05 epoch=0.025806451612903226
[step 6/194] loss=8.555482864379883 lr=5e-05 epoch=0.03096774193548387
[step 7/194] loss=7.616472244262695 lr=6e-05 epoch=0.03612903225806452
[step 8/194] loss=7.056985855102539 lr=7e-05 epoch=0.04129032258064516
[step 9/194] loss=15.987354278564453 lr=8e-05 epoch=0.04645161290322581
[step 10/194] loss=9.526388168334961 lr=9e-05 epoch=0.05161290322580645
[step 11/194] loss=15.047636985778809 lr=0.0001 epoch=0.0567741935483871
[step 12/194] loss=7.241550922393799 lr=0.00011000000000000002 epoch=0.06193548387096774
[step 13/194] loss=9.880370140075684 lr=0.00012 epoch=0.06709677419354838
[step 14/194] loss=8.405094146728516 lr=0.00013000

TrainOutput(global_step=194, training_loss=5.0305360985785414, metrics={'train_runtime': 2352.1852, 'train_samples_per_second': 0.329, 'train_steps_per_second': 0.082, 'total_flos': 1.99064707344576e+16, 'train_loss': 5.0305360985785414, 'epoch': 1.0})

In [14]:
# Save LoRA adapter files
print("=== Saving Adapter ===")
trainer.model.save_pretrained(OUTPUT_DIR)

print(f"Saved adapter files to: {OUTPUT_DIR}")

saved_files = sorted(
    f for f in os.listdir(OUTPUT_DIR)
    if os.path.isfile(os.path.join(OUTPUT_DIR, f))
)

for fname in saved_files:
    fpath = os.path.join(OUTPUT_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  - {fname} ({size_kb:.1f} KB)")

=== Saving Adapter ===
Saved adapter files to: /kaggle/working/lora_adapter
  - README.md (5.2 KB)
  - adapter_config.json (1.1 KB)
  - adapter_model.safetensors (3454393.7 KB)


In [15]:
# Create submission.zip with files placed at the zip root
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fname in saved_files:
        fpath = os.path.join(OUTPUT_DIR, fname)
        zf.write(fpath, arcname=fname)

zip_size_mb = os.path.getsize(ZIP_PATH) / 1024 / 1024
print(f"\nCreated submission archive: {ZIP_PATH} ({zip_size_mb:.2f} MB)")

# Verify required files
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    archive_files = sorted(zf.namelist())

print("Archive contents:")
for fname in archive_files:
    print(f"  - {fname}")

required_files = ["adapter_config.json"]
missing_files = [fname for fname in required_files if fname not in archive_files]

if missing_files:
    raise ValueError(f"Missing required file(s) in submission.zip: {missing_files}")

print("\nSubmission package looks valid and is ready for upload.")


Created submission archive: /kaggle/working/submission.zip (3008.02 MB)
Archive contents:
  - README.md
  - adapter_config.json
  - adapter_model.safetensors

Submission package looks valid and is ready for upload.
